# 01 — Preprocessing

Builds the hourly modelling table from the raw files in `data/raw/`:

- **OMNI2 hourly** solar wind (features) → fill sentinels to NaN, UTC time index
- **GFZ Kp** (label) → Kp1–Kp8 melted to 3-hourly blocks, definitive flag kept
- Complete hourly grid; each hour gets the Kp of the 3-h block containing it

Output: `data/processed/tormenta_hourly.csv` and `data/f107_history.csv`.
All logic lives in `src/` so the live apps reuse it; this notebook only runs and inspects it.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.ingest.fetch_omni import load_omni_lst
from src.ingest.fetch_gfz import load_gfz_daily, kp_3hourly
from src import preprocessing as pp

## 1. Load OMNI

In [2]:
omni = load_omni_lst()
print(omni.shape, omni.index.min(), '→', omni.index.max())
omni.head()

(245688, 14) 1998-01-01 00:00:00 → 2026-01-10 23:00:00


,b_scalar,bx,by_gsm,bz_gsm,temperature,density,speed,omni_pressure,omni_efield,omni_beta,omni_mach_a,omni_mach_ms,sunspot_r,f107
time,,,,,,,,,,,,,,
1998-01-01 00:00:00,4.6,-1.5,1.4,2.2,105256.0,6.3,364.0,1.48,-0.80,2.89,9.9,5.3,54,98.3
1998-01-01 01:00:00,4.7,-2.3,-1.0,3.3,109322.0,7.0,365.0,1.67,-1.20,3.13,10.3,5.3,54,98.3
1998-01-01 02:00:00,5.0,-4.3,-0.1,-0.9,86514.0,4.6,350.0,1.13,0.32,1.64,7.5,4.8,54,98.3
1998-01-01 03:00:00,5.5,-4.5,-1.1,-1.1,97310.0,5.8,357.0,1.48,0.39,1.80,7.8,4.9,54,98.3
1998-01-01 04:00:00,5.9,-5.2,0.0,-1.2,89358.0,5.2,355.0,1.41,0.43,1.35,6.9,4.7,54,98.3


## 2. Load GFZ Kp

In [3]:
gfz = load_gfz_daily()
kp3 = kp_3hourly(gfz)
print('GFZ days:', len(gfz), '| 3-h blocks:', len(kp3))
print('Non-definitive days (D≠2):', (~gfz['definitive']).sum())
kp3.tail(10)

GFZ days: 34632 | 3-h blocks: 277056
Non-definitive days (D≠2): 117


,kp,definitive
time,,
2026-10-24 18:00:00,1.333,False
2026-10-24 21:00:00,0.667,False
2026-10-25 00:00:00,0.333,False
2026-10-25 03:00:00,0.667,False
2026-10-25 06:00:00,0.000,False
2026-10-25 09:00:00,0.667,False
2026-10-25 12:00:00,0.667,False
2026-10-25 15:00:00,0.333,False
2026-10-25 18:00:00,1.000,False


## 3. Merge into the hourly table

In [4]:
df = pp.build_hourly(omni, gfz)
pp.check(df)
df.head()

Sanity checks passed: unique timestamps, complete hourly grid, Kp in [0, 9].


,b_scalar,bx,by_gsm,bz_gsm,temperature,density,speed,omni_pressure,omni_efield,omni_beta,omni_mach_a,omni_mach_ms,kp,kp_definitive,f107_obs
time,,,,,,,,,,,,,,,
1998-01-01 00:00:00,4.6,-1.5,1.4,2.2,105256.0,6.3,364.0,1.48,-0.80,2.89,9.9,5.3,0.667,True,101.6
1998-01-01 01:00:00,4.7,-2.3,-1.0,3.3,109322.0,7.0,365.0,1.67,-1.20,3.13,10.3,5.3,0.667,True,101.6
1998-01-01 02:00:00,5.0,-4.3,-0.1,-0.9,86514.0,4.6,350.0,1.13,0.32,1.64,7.5,4.8,0.667,True,101.6
1998-01-01 03:00:00,5.5,-4.5,-1.1,-1.1,97310.0,5.8,357.0,1.48,0.39,1.80,7.8,4.9,1.333,True,101.6
1998-01-01 04:00:00,5.9,-5.2,0.0,-1.2,89358.0,5.2,355.0,1.41,0.43,1.35,6.9,4.7,1.333,True,101.6


## 4. Summary: rows, columns, missing %

In [5]:
pp.summarize(df)

Rows (hours): 245,688   Columns: 15
Span: 1998-01-01 00:00:00 → 2026-01-10 23:00:00
Rows with definitive Kp label: 245,688 (~81,896 independent 3-h labels)


,missing_%
b_scalar,0.15
bx,0.15
by_gsm,0.16
bz_gsm,0.16
temperature,2.05
density,2.27
speed,0.37
omni_pressure,2.28
omni_efield,0.41
omni_beta,2.34


## 5. Spot check: May 2024 (Gannon) storm
Strongly southward Bz with fast wind should line up with Kp 8–9.

In [6]:
df.loc['2024-05-10 12:00':'2024-05-11 12:00', ['bz_gsm', 'speed', 'density', 'kp']]

,bz_gsm,speed,density,kp
time,,,,
2024-05-10 12:00:00,-1.9,409.0,2.9,3.667
2024-05-10 13:00:00,0.6,411.0,3.2,3.667
2024-05-10 14:00:00,-1.0,446.0,5.1,3.667
2024-05-10 15:00:00,-0.6,455.0,14.2,7.667
2024-05-10 16:00:00,1.9,449.0,15.6,7.667
2024-05-10 17:00:00,-5.9,571.0,29.1,7.667
2024-05-10 18:00:00,-18.0,697.0,32.9,8.667
2024-05-10 19:00:00,-1.2,664.0,35.2,8.667
2024-05-10 20:00:00,-26.9,702.0,48.1,8.667


## 6. Save

In [7]:
pp.save_processed(df)
hist = pp.build_f107_history(gfz)
print('Saved', config.PROCESSED_CSV.relative_to(ROOT))
print('Saved', config.F107_HISTORY.relative_to(ROOT), f'({len(hist)} days)')

Saved data\processed\tormenta_hourly.csv
Saved data\f107_history.csv (364 days)
